In [ ]:
import allel
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go  # type: ignore

import warnings
warnings.filterwarnings('ignore')


In [ ]:
metadata_path = '../../../results/config/metadata.qcpass.tsv'
bed_targets_path = "../../../config/ag-vampir.bed"
vcf_path = "../../..//results/vcfs/targets/lab-strains.annot.vcf"
wkdir = "../../.."
cohort_cols = 'location,taxon'
platform = 'illumina'


In [ ]:
# Parameters
platform = "illumina"
dataset = "ag-vampir-combined"
metadata_path = "results/config/metadata.qcpass.tsv"
wkdir = "/mnt/user_shares/snagi/lstm_scratch/network_scratch/ampseq-agvampir-combined"
vcf_path = "results/vcfs/targets/ag-vampir-combined.annot.vcf"
bed_targets_path = "config/ag-vampir.bed"
cohort_cols = "location,taxon,dataset_origin"


In [ ]:
import os
import sys

sys.path.append(os.path.join(wkdir, 'workflow/lib'))
import ampseeker as amp


### Taxon classification

In this analysis, we use three separate methods (AIMs, PCA, XGboost) to assign taxon to amplicon samples. For the latter two methods, we integrate data from over 7000 public accessible WGS samples from the [Vector Observatory](https://www.malariagen.net/vobs/).

Accurate species identification is essential for mosquito surveillance as different members of the Anopheles gambiae complex have distinct ecological niches and vectorial capacities (Coetzee et al., 2013).

#### Ancestry informative marker heatmap

In [ ]:
cohort_cols_list = [c.strip() for c in cohort_cols.split(',') if c.strip()]
cohort_col = cohort_cols_list[0]

metadata = pd.read_csv(metadata_path , sep="\t")

# Avoid grouping and counting on the same taxon column in summary tables.
summary_cohort_col = cohort_col
if summary_cohort_col == 'taxon':
    summary_cohort_col = next((c for c in cohort_cols_list if c != 'taxon' and c in metadata.columns), 'sample_id')

import json
with open(f"{wkdir}/results/config/metadata_colours.json", 'r') as f:
    color_mapping = json.load(f)
    
targets = pd.read_csv(bed_targets_path, sep="\t", header=None)
targets.columns = ['contig', 'start', 'end', 'amplicon', 'mutation', 'ref', 'alt']

gn_amp, pos, gn_contigs, metadata, refs, alts_amp, ann = amp.load_variants(vcf_path=vcf_path, metadata=metadata, platform=platform, filter_indel=True)
samples = metadata['sample_id'].to_list()

alts_amp = np.concatenate([refs.reshape(refs.shape[0], -1), alts_amp], axis=1)


In [ ]:
contigs = ['2R', '2L', '3R', '3L', 'X']
df_aims = targets.query("mutation.str.contains('AIM')", engine='python')

aim_mask = np.isin(pos, df_aims.end.to_list())
aim_gn = gn_amp.compress(aim_mask, axis=0)
aim_pos = pos[aim_mask]
aim_contigs = gn_contigs[aim_mask]
aim_alts = alts_amp[aim_mask]

aim_loc = ["aim_" + c + ":" + str(aim_pos[i]) for i, c in enumerate(aim_contigs)]
df_aims = df_aims.assign(loc=lambda x: "aim_" + x.contig + ":" + x.end.astype(str)).set_index('loc')
df_aims = df_aims.loc[aim_loc]

aim_gn_alt = amp.aims_n_alt(aim_gn, aim_alts=df_aims.alt.to_list(), data_alts=aim_alts)
df_aims = pd.concat([df_aims, pd.DataFrame(aim_gn_alt, columns=samples, index=aim_loc)], axis=1)
df_aims = pd.concat([df_aims.query(f"contig == '{contig}'") for contig in contigs])

# Order rows by the colour-map order (expected species group: gambiae -> coluzzii -> arabiensis), then strain / cohort,
# then by mean AIM genotype (gambiae-like to coluzzii-like). Cohorts not listed go last.
strain_order = list(color_mapping[cohort_col]) if cohort_col in color_mapping else []
cohorts_present = list(metadata[cohort_col].unique())
cohort_plot_order = [s for s in strain_order if s in cohorts_present] + [s for s in cohorts_present if s not in strain_order]

aim_gn_mean = df_aims.iloc[:, 7:].replace({-1: np.nan}).mean()
aimplot_sample_order = []
row_cohort = []
for coh in cohort_plot_order:
    coh_samples = metadata.query(f"{cohort_col} == @coh").sample_id.to_list()
    coh_order = aim_gn_mean.loc[coh_samples].sort_values(ascending=True).index.to_list()
    if aimplot_sample_order:
        aimplot_sample_order.append(f"gap_{coh}")   # blank spacer row between cohorts
        row_cohort.append(None)
    aimplot_sample_order.extend(coh_order)
    row_cohort.extend([coh] * len(coh_order))

from plotly.subplots import make_subplots
col_widths = [1.5] + [np.count_nonzero(aim_contigs == contig) for contig in contigs]

fig = make_subplots(
    rows=1,
    cols=len(contigs) + 1,
    shared_yaxes=True,
    column_titles=[""] + contigs,
    column_widths=col_widths,
    horizontal_spacing=0.008,
)

# Horizontal legend below the plot, species names in italics.
colorbar = dict(
    tickmode="array",
    tickvals=[-1, 0, 1, 2],
    ticktext=["Missing", "<i>An. gambiae</i>", "Heterozygote", "<i>An. coluzzii</i>"],
    orientation="h",
    x=0.55, xanchor="center", y=-0.005, yanchor="top",
    len=0.7, lenmode="fraction", thickness=14,
    outlinewidth=1, outlinecolor="#999999",
    tickfont=dict(size=13),
)

# AIMs palette matches manuscript Fig 2
palette = ("white", "#DDCC77", "#F9E4CC", "#CC6677")
colorscale = [
    [0 / 4, palette[0]], [1 / 4, palette[0]],
    [1 / 4, palette[1]], [2 / 4, palette[1]],
    [2 / 4, palette[2]], [3 / 4, palette[2]],
    [3 / 4, palette[3]], [4 / 4, palette[3]],
]

# Cohort colour strip (first column)
n_coh = len(cohort_plot_order)
coh_idx = {coh: i for i, coh in enumerate(cohort_plot_order)}
strip_z = [[np.nan if c is None else coh_idx[c]] for c in row_cohort]
coh_cols = [color_mapping.get(cohort_col, {}).get(coh, "#999999") for coh in cohort_plot_order]
strip_scale = []
for i, col in enumerate(coh_cols):
    strip_scale += [[i / n_coh, col], [(i + 1) / n_coh, col]]
fig.add_trace(
    go.Heatmap(y=aimplot_sample_order, z=strip_z, x=[0], colorscale=strip_scale,
               zmin=-0.5, zmax=n_coh - 0.5, showscale=False, hoverinfo="skip"),
    row=1, col=1,
)

# One heatmap per contig
for j, contig in enumerate(contigs):
    df_c = df_aims.filter(like=contig, axis=0).iloc[:, 7:]
    df_c = df_c.T.reindex(aimplot_sample_order)   # spacer rows become NaN (blank)
    fig.add_trace(
        go.Heatmap(
            y=df_c.index, z=df_c, x=df_c.columns,
            colorscale=colorscale, zmin=-1.5, zmax=2.5, xgap=0, ygap=0,
            showscale=(j == 0), colorbar=colorbar,
        ),
        row=1, col=j + 2,
    )

# Cohort labels at the centre of each block
for coh in cohort_plot_order:
    rows = [s for s, c in zip(aimplot_sample_order, row_cohort) if c == coh]
    fig.add_annotation(
        x=0, xref="paper", xanchor="right", xshift=-4,
        y=rows[len(rows) // 2], yref="y", showarrow=False,
        text=coh.replace("_", " "), font=dict(size=13, color="#333333"),
    )

fig.update_yaxes(autorange="reversed", showticklabels=False, showgrid=False, zeroline=False)
fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False)
fig.update_layout(
    height=max(700, 1.1 * len(aimplot_sample_order) + 180),
    width=1000,
    margin=dict(l=110, r=30, t=60, b=90),
    plot_bgcolor="white", paper_bgcolor="white",
)
fig.write_image(f"{wkdir}/results/aims_gamb_vs_colu.png", scale=2)
fig.write_image(f"{wkdir}/results/aims_gamb_vs_colu.svg")

fig.show()


#### By cohort

Ancestry Informative Markers (AIMs) are genomic variants with large allele frequency differences between populations, making them useful for species identification and detecting hybridization (Rosenberg et al., 2003).

In [ ]:
df = df_aims.iloc[:, 7:]
df.columns = samples

## Use only chrom 3 and X aims for taxon 
x_3_mask = df.index.str.contains("aim_3|aim_X")
df = df[x_3_mask]

mean_aims = df.replace(-1, float('nan')).apply(np.nanmean, axis=0)
max_missing_aims = 12
mean_aims[df.replace(-1, float('nan')).isna().sum(axis=0) > max_missing_aims] = np.nan
aims = mean_aims.loc[metadata.set_index('sample_id').index]
metadata = metadata.assign(mean_aim_genotype=aims.values)

taxon = []
for i, row in metadata.iterrows():
    if row.mean_aim_genotype == np.nan:
        taxon.append('uncertain')
    elif row.mean_aim_genotype < 0.5:
        taxon.append('gambiae')
    elif row.mean_aim_genotype >= 0.5 and row.mean_aim_genotype < 1.5:
        taxon.append('unassigned')
    elif row.mean_aim_genotype >= 1.5:
        taxon.append('coluzzii')
    else: 
        taxon.append(np.nan)

new_metadata = metadata.assign(aim_taxon=taxon)

fig = px.histogram(
    new_metadata,
    nbins=100, 
    x='mean_aim_genotype', 
    color=cohort_col, 
    color_discrete_map=color_mapping[cohort_col],
    category_orders={cohort_col: list(color_mapping[cohort_col])},
    width=750, 
    height=400, 
    template='plotly_white', 
    title='AIM genotype distribution'
    )
fig.show()


In [ ]:
from IPython.display import display, Markdown
new_metadata[['sample_id', 'aim_taxon', 'mean_aim_genotype']].to_csv(f"{wkdir}/results/ag-vampir/aims/taxon_aims.tsv", sep="\t", index=False)
display(Markdown(f'<a href={wkdir}/results/ag-vampir/aims/taxon_aims.tsv>Sample gambiae vs coluzzii AIMs and AIM taxon assignment (.tsv)</a>'))


### PCA-based taxon classification

In this method, we use perform PCA and then apply a classifier to predict taxon based on VObs reference data. 

Principal Component Analysis reduces the dimensionality of genetic data while preserving patterns that differentiate species, enabling classification through machine learning algorithms (Jombart et al., 2010).

In [ ]:
### Load VObs PCA data
ampseq_loci = np.load(f"{wkdir}/resources/ag-vampir/agvampir-ampseq-loci.npy")
alts_wgs = np.load(f"{wkdir}/resources/ag-vampir/agvampir-alts-wgs.npy")
gn_wgs = np.load(f"{wkdir}//resources/ag-vampir/agvampir-gn-wgs.npy")

df_wgs_samples = pd.read_csv(f"{wkdir}/resources/ag-vampir/agvampir-df-samples.csv", index_col=0)
df_wgs_samples.shape
    
targets = pd.read_csv(bed_targets_path, sep="\t", header=None)
targets.columns = ['contig', 'start', 'end', 'amplicon', 'mutation', 'ref', 'alt']

gn_amp, pos, gn_contigs, metadata, refs, alts_amp, ann = amp.load_variants(vcf_path=vcf_path, metadata=new_metadata, platform=platform, filter_indel=True)
samples = metadata['sample_id'].to_list()

alts = np.concatenate([refs.reshape(refs.shape[0], -1), alts_amp], axis=1)

wgs_pos  = np.array([int(s.split(":")[1].split("-")[0]) for s in ampseq_loci])

wgs_dict = {value: idx for idx, value in enumerate(wgs_pos)}
indices = [wgs_dict.get(value, -1) for value in pos]
valid_indices = [idx for idx in indices if idx != -1]
wgs_pos = np.array([wgs_pos[i] for i in valid_indices])
gn_wgs = gn_wgs.take(valid_indices, axis=0)
alts_wgs = alts_wgs.take(valid_indices, axis=0)

assert all(np.array(wgs_pos) == pos)


In [ ]:
mapping = allel.create_allele_mapping(alts_wgs[:, 0], alt=alts_amp, alleles=alts_wgs)
gn_amp_remap = gn_amp.map_alleles(mapping)

snp_miss_mask = gn_amp_remap.is_missing().sum(axis=1) > 40

gn_amp_flt = gn_amp_remap.compress(~snp_miss_mask, axis=0)
gn_wgs_flt = gn_wgs.compress(~snp_miss_mask, axis=0)

mask = df_wgs_samples.eval("taxon not in ['unassigned', 'fontenillei', 'gcx4', 'quadriannulatus', 'merus', 'melas']")
df_wgs_samples_flt = df_wgs_samples[mask]
gn_wgs_flt = gn_wgs_flt.compress(mask, axis=1)

pca_df = amp.pca_all_samples(gn_wgs_flt, gn_amp_flt, df_wgs_samples_flt, metadata, n_components=6, cohort_cols=cohort_cols)

df_assignments = amp.assign_taxa(pca_df, method="svm", n_neighbors=5, 
                           probability_threshold=0.8)

metadata = metadata.assign(pca_taxon=df_assignments.predicted_taxon.to_numpy())

pca_3d = amp.plot_pca_3d_with_assignments(
        pca_df, 
        df_assignments, 
        title="3D PCA with Taxon Assignments"
)

pca_3d

### Machine learning based taxon classification

In this method, we use an XGBoost algorithm to predict species based on the VObs reference data. 

XGBoost is a gradient boosting framework that uses decision trees to identify the most informative features for classification, offering high accuracy for species assignment (Chen & Guestrin, 2016).

In [ ]:
import pickle
import xgboost 

with open(f"{wkdir}/resources/ag-vampir/xgboost_taxon.pickle", 'rb') as file:
        tree = pickle.load(file)
taxon_tree_features = tree['feature_importance'].sort_index().SNP.to_list()
taxon_tree_feat_pos = np.unique([int(t.split(":")[1].split("-")[1]) for t in taxon_tree_features])

aim_gn = gn_amp.copy()
aim_pos = pos.copy()


In [ ]:
#All nucleotides that should be in each row
all_nucleotides = set(['A', 'C', 'G', 'T'])

arr = alts.copy()

# Fill in the missing values
for i in range(arr.shape[0]):
    # Get the non-empty values in the current row
    present = set([val for val in arr[i] if val != ''])
    
    # Find the missing nucleotides
    missing = all_nucleotides - present
    
    # Fill in the missing values
    j = 0
    for k in range(arr.shape[1]):
        if arr[i, k] == '':
            if j < len(missing):
                arr[i, k] = list(missing)[j]
                j += 1


In [ ]:
gn_counts = amp._melt_gt_counts(gn_amp.to_allele_counts(max_allele=3).values)

alts_flat = arr[:, 1:].flatten()
pos_flat = np.repeat(pos, 3)

labels = np.array([f"{np.repeat(gn_contigs, 3)[i]}:{pos_flat[i]}-{pos_flat[i]}:{alts_flat[i]}" for i in range(len(alts_flat))])

df_vampir = pd.DataFrame(gn_counts, index=labels, columns=samples)
missing_features = [feature for feature in taxon_tree_features if feature not in df_vampir.index]
if missing_features:
    missing_df = pd.DataFrame(0, index=missing_features, columns=df_vampir.columns)
    df_vampir = pd.concat([df_vampir, missing_df])
    print(f"WARNING: The following features were missing and artificially added with zeros: {missing_features}")
    print("Taxon assignment using XGBoost may not be accurate due to these missing features.")

df_vampir = df_vampir.loc[taxon_tree_features, :]

taxon_labels = list(tree['per_taxon_results'].keys())
pred = tree['model'].predict(df_vampir.T)
metadata = metadata.assign(tree_taxon=tree['label_encoder'].inverse_transform(pred))

pivot_table = metadata.pivot_table(
    index=summary_cohort_col,
    columns='tree_taxon',
    values='sample_id',  
    aggfunc='count',
    fill_value=0
)
pivot_table


### Consensus taxon calls summary

Consensus taxonomic assignment combines predictions from multiple methods, reducing errors and improving confidence in species identification.

In [ ]:
# Apply the function to create the new column
metadata['taxon'] = metadata.apply(amp.get_consensus_taxon, axis=1)

metadata.to_csv(f"{wkdir}/results/config/metadata.qcpass.tsv", sep="\t", index=False)

pivot_table = metadata.pivot_table(
    index=summary_cohort_col,
    columns='taxon',
    values='sample_id',  
    aggfunc='count',
    fill_value=0
)
pivot_table


In [ ]:
pd.set_option("display.max_rows", 500)

cohort_cols_header = cohort_cols.split(',')
if 'taxon' in cohort_cols_header:
    cohort_cols_header.remove('taxon')

show_cols = ['sample_id'] + cohort_cols_header + ['mean_aim_genotype', 'aim_taxon', 'pca_taxon', 'tree_taxon', 'taxon']

metadata[show_cols]
